# Fundamentos de IA — Demos

Notebook con las cuatro demos de la clase, pensado para correr **en vivo, celda por celda**. Todo es local, sin claves de API.

| # | Demo | Bloque | Dura |
|---|------|--------|------|
| 1 | Descenso por gradiente | 3 — Redes y backprop | ~3 min |
| 2 | Embeddings y similitud | 4 — Embeddings | ~3 min |
| 3 | Tokenización (BPE) | 5 — Transformers | ~2 min |
| 4 | Matriz de atención | 5 — Transformers | ~3 min |

> **Antes de la clase:** corré la celda de setup y, sobre todo, la **Demo 2** una vez — la primera ejecución baja el modelo de embeddings (~80 MB) y conviene tenerlo cacheado para que no rompa el flujo en vivo.

## Setup

Si ya tenés las dependencias instaladas en el kernel, saltá esta celda. Si no, descomentá y corré una vez.

In [ ]:
# !pip install tiktoken sentence-transformers matplotlib numpy

# Para que los plots de matplotlib se rendericen dentro del notebook:
%matplotlib inline

---
## Demo 1 — Descenso por gradiente (Bloque 3)

Soltamos una "pelotita" lejos del mínimo de una cuadrática. En cada paso calculamos el gradiente (la derivada en ese punto) y movemos `x` un pasito **en contra** del gradiente. La pelotita baja, la loss baja.

Esto es backprop conceptualmente: en una red real pasa en miles de millones de dimensiones a la vez, pero la idea es exactamente la misma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Función a minimizar: una cuadrática con mínimo en x=3, f=2.
def f(x):
    return (x - 3) ** 2 + 2

# Gradiente analítico: f'(x) = 2(x - 3).
def grad(x):
    return 2 * (x - 3)

# Hiperparámetros.
x = -4.0          # arrancamos lejos del mínimo
lr = 0.1          # learning rate (tamaño del pasito)
steps = 30        # cuántas iteraciones

# Historial para graficar.
xs, ys = [x], [f(x)]
for _ in range(steps):
    x = x - lr * grad(x)   # paso de descenso por gradiente
    xs.append(x)
    ys.append(f(x))

print(f"x final:    {x:.4f}  (mínimo real: 3)")
print(f"loss final: {f(x):.4f}  (mínimo real: 2)")

# Visualización.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: la función y la trayectoria de la pelotita.
xx = np.linspace(-5, 8, 200)
ax1.plot(xx, f(xx), label="f(x) = (x-3)^2 + 2")
ax1.scatter(xs, ys, c="red", s=30, label="trayectoria")
ax1.set_title("Descenso por gradiente sobre f(x)")
ax1.set_xlabel("x (la 'perilla')"); ax1.set_ylabel("loss")
ax1.legend()

# Panel 2: la loss a lo largo de las iteraciones.
ax2.plot(ys, marker="o")
ax2.set_title("Loss vs iteración")
ax2.set_xlabel("iteración"); ax2.set_ylabel("loss")

plt.tight_layout()
plt.show()

---
## Demo 2 — Embeddings y similitud coseno (Bloque 4)

Tres frases: dos son sobre mascotas, una es burocracia AFIP. Tras pasarlas por el modelo, cada una es un vector de 384 dimensiones. Calculamos coseno entre todos los pares.

Las dos de mascotas dan score alto entre sí; cualquiera contra la de AFIP da score bajo. **La geometría refleja el significado** — esto es lo que hace funcionar a RAG.

> La primera ejecución descarga `all-MiniLM-L6-v2` (~80 MB) y lo cachea en `~/.cache/`.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo chiquito que corre en CPU, embeddings de 384 dimensiones.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

frases = [
    "El perro corre por el parque.",
    "El gato duerme en el sillón.",
    "Factura electrónica AFIP comprobante tipo A.",
]

# encode() devuelve un array (N, 384). normalize_embeddings=True hace que
# el producto punto sea directamente la similitud coseno.
emb = model.encode(frases, normalize_embeddings=True)
print("Shape del embedding:", emb.shape)

# Similitud coseno = producto punto entre vectores normalizados.
sim = emb @ emb.T

print("\nMatriz de similitud coseno:")
header = " | ".join(f"frase {i}" for i in range(len(frases)))
print(f"{'':40}{header}")
for i, fr in enumerate(frases):
    fila = "  ".join(f"{x:+.3f}" for x in sim[i])
    print(f"frase {i}: {fr[:38]:38}  {fila}")

In [ ]:
# Opcional: la misma matriz como heatmap, más visual para la clase.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
labels = [f"frase {i}" for i in range(len(frases))]
ax.set_xticks(range(len(frases))); ax.set_xticklabels(labels)
ax.set_yticks(range(len(frases))); ax.set_yticklabels(labels)
for i in range(len(frases)):
    for j in range(len(frases)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="white")
ax.set_title("Similitud coseno entre frases")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

---
## Demo 3 — Tokenización BPE (Bloque 5)

Cómo `tiktoken` (el tokenizador BPE oficial de OpenAI) parte el texto. No es "una palabra = un token": las palabras comunes son un token, las raras o en otros idiomas se parten en subwords.

Mirá la diferencia entre inglés y español: el vocabulario fue entrenado dominantemente en inglés, así que `andás` se rompe en pedacitos. Esto importa porque **cobran por token** y el **context window se mide en tokens**.

In [ ]:
import tiktoken

# Encoding o200k_base: el usado por GPT-4o y modelos posteriores.
enc = tiktoken.get_encoding("o200k_base")

ejemplos = [
    "Hello, world!",
    "Hola, mundo, ¿cómo andás?",
    "MLOps developers love Docker images.",
    "Los desarrolladores MLOps aman las imágenes de Docker.",
]

for texto in ejemplos:
    ids = enc.encode(texto)
    # decode_single_token_bytes devuelve los bytes de cada token.
    pieces = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace")
              for t in ids]
    print(f"\nTexto:   {texto!r}")
    print(f"Tokens:  {pieces}")
    print(f"IDs:     {ids}")
    print(f"Total:   {len(ids)} tokens")

---
## Demo 4 — Matriz de atención (Bloque 5)

Una matriz de atención **simulada** (no de un modelo real, sino armada a mano para ilustrar el patrón). Cada fila es un token preguntando "¿a quién miro?"; cada columna, un token siendo mirado. Celdas más claras = atención más fuerte.

Fijate: el verbo `ajusta` atiende al sujeto `modelo`; `función`, `loss` y `gradiente` se miran mutuamente porque conceptualmente van juntos. En un Transformer real esto **emerge solo del entrenamiento**, con muchas capas y muchas cabezas, cada una con su patrón.

> *Esta es la demo salteable si vas corto de tiempo.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tokens = ["El", "modelo", "ajusta", "la", "función",
          "de", "loss", "con", "el", "gradiente", "."]
n = len(tokens)

# Matriz random suave + refuerzos en relaciones específicas.
rng = np.random.default_rng(42)
A = rng.uniform(0.0, 0.15, size=(n, n))

# Refuerzos didácticos (fila = quien atiende, columna = a quién):
A[2, 1] += 0.7   # 'ajusta' -> 'modelo'  (verbo -> sujeto)
A[4, 6] += 0.7   # 'función' -> 'loss'
A[6, 4] += 0.5   # 'loss' -> 'función'
A[9, 4] += 0.6   # 'gradiente' -> 'función'
A[9, 6] += 0.5   # 'gradiente' -> 'loss'
A[3, 4] += 0.6   # 'la' -> 'función' (artículo -> sustantivo)
A[8, 9] += 0.6   # 'el' -> 'gradiente'

# Cada fila suma 1 (softmax-like), como en atención real.
A = A / A.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(A, cmap="viridis")
ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(tokens)
ax.set_xlabel("Atendido (key)")
ax.set_ylabel("Atendiendo (query)")
ax.set_title("Matriz de atención (ilustrativa)\nfila = token que mira, columna = token mirado")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

---
**Fin de las demos.** Vuelta a la presentación (`presentacion.html`).